In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
csv_path = Path("/path/to/project/variation_in_wind_drift_relation/alpha_theta_grouped.csv")
df = pd.read_csv(csv_path)

print(df.head())
print(df["split"].unique())
print(df["group_type"].unique())

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D

TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27


def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })


def fig_textwidth(height_ratio=0.62):
    return (TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio)


def plot_alpha_theta_by_year(
    df,
    splits=("train", "val", "test"),
    exclude_years=(2014,),
    fontsize=9,
    height_ratio=0.62,
    save_path=None,
):
    setup_pub_style(fontsize=fontsize)
    sns.set_style("whitegrid")

    palette = sns.color_palette("colorblind")
    alpha_color = palette[0]
    theta_color = palette[1]

    fig, ax1 = plt.subplots(figsize=fig_textwidth(height_ratio))
    ax2 = ax1.twinx()

    style_map = {
        "train": {"linestyle": "-",  "marker": "o"},
        "val":   {"linestyle": "--", "marker": "s"},
        "test":  {"linestyle": ":",  "marker": "^"},
    }

    # Prepare cleaned yearly data for each split
    split_data = {}
    for split in splits:
        d = df[(df["split"] == split) & (df["group_type"] == "year")].copy()
        if d.empty:
            continue

        d["calendar_year"] = d["calendar_year"].astype(int)
        d = d[~d["calendar_year"].isin(exclude_years)]
        d = d.sort_values("calendar_year").reset_index(drop=True)

        if not d.empty:
            split_data[split] = d

    valid_splits = [s for s in splits if s in split_data]

    for i, split in enumerate(valid_splits):
        d = split_data[split]

        ls = style_map.get(split, {"linestyle": "-", "marker": "o"})["linestyle"]
        mk = style_map.get(split, {"linestyle": "-", "marker": "o"})["marker"]

        # Always connect current split to first point of next split, if it exists
        next_d = None
        if i + 1 < len(valid_splits):
            next_d = split_data[valid_splits[i + 1]]

        if next_d is not None and not next_d.empty:
            d_line = pd.concat([d, next_d.head(1)], ignore_index=True)
        else:
            d_line = d

        # Alpha
        ax1.plot(
            d_line["calendar_year"],
            d_line["alpha"],
            color=alpha_color,
            linestyle=ls,
            marker=None,
            linewidth=1.5,
        )
        ax1.plot(
            d["calendar_year"],
            d["alpha"],
            color=alpha_color,
            linestyle="None",
            marker=mk,
            markersize=4,
        )

        # Theta
        ax2.plot(
            d_line["calendar_year"],
            d_line["theta_deg"],
            color=theta_color,
            linestyle=ls,
            marker=None,
            linewidth=1.5,
        )
        ax2.plot(
            d["calendar_year"],
            d["theta_deg"],
            color=theta_color,
            linestyle="None",
            marker=mk,
            markersize=4,
        )

    ax1.set_xlabel("Year")
    ax1.set_ylabel(r"$\alpha$", color=alpha_color)
    ax2.set_ylabel(r"$\theta$ [$^\circ$]", color=theta_color)

    ax1.tick_params(axis="y", colors=alpha_color)
    ax2.tick_params(axis="y", colors=theta_color)

    ax1.grid(True, which="major", alpha=0.3)
    ax2.grid(False)

    split_handles = []
    for split in valid_splits:
        if split == "test":
            # marker only in legend
            split_handles.append(
                Line2D(
                    [0], [0],
                    color="0.2",
                    linestyle="None",
                    marker=style_map[split]["marker"],
                    markersize=4,
                    label=split.capitalize(),
                )
            )
        else:
            split_handles.append(
                Line2D(
                    [0], [0],
                    color="0.2",
                    linestyle=style_map[split]["linestyle"],
                    marker=style_map[split]["marker"],
                    lw=1.5,
                    markersize=4,
                    label=split.capitalize(),
                )
            )

    ax1.legend(
        handles=split_handles,
        handlelength=2.5,
        loc="upper left",
        title="Split",
        frameon=True,
    )

    ax1.set_ylim(0.02, 0.028)
    ax2.set_ylim(18, 24)

    sns.despine(ax=ax1, right=False)
    sns.despine(ax=ax2, left=False)

    fig.tight_layout()
    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()

In [ ]:
plot_alpha_theta_by_year(df, exclude_years=(2014,), height_ratio=0.35, save_path="alpha_theta_by_year.pdf")

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.lines import Line2D

TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27


def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })


def fig_textwidth(height_ratio=0.62):
    return (TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio)


def plot_alpha_theta_by_season(
    df,
    splits=("train", "val", "test"),
    fontsize=9,
    height_ratio=0.62,
    save_path=None,
):
    setup_pub_style(fontsize=fontsize)
    sns.set_style("whitegrid")

    palette = sns.color_palette("colorblind")
    alpha_color = palette[0]
    theta_color = palette[1]

    fig, ax1 = plt.subplots(figsize=fig_textwidth(height_ratio))
    ax2 = ax1.twinx()

    season_order = ["DJF", "MAM", "JJA", "SON"]
    x_pos = list(range(len(season_order)))

    style_map = {
        "train": {"linestyle": "-",  "marker": "o"},
        "val":   {"linestyle": "--", "marker": "s"},
        "test":  {"linestyle": ":",  "marker": "^"},
    }

    for split in splits:
        d = df[(df["split"] == split) & (df["group_type"] == "season")].copy()
        if d.empty:
            continue

        d["season"] = pd.Categorical(d["season"], categories=season_order, ordered=True)
        d = d.sort_values("season").reset_index(drop=True)

        # keep only rows with valid seasons in the desired order
        d = d.dropna(subset=["season"])
        if d.empty:
            continue

        ls = style_map.get(split, {"linestyle": "-", "marker": "o"})["linestyle"]
        mk = style_map.get(split, {"linestyle": "-", "marker": "o"})["marker"]

        split_x = [season_order.index(s) for s in d["season"].astype(str)]

        ax1.plot(
            split_x,
            d["alpha"],
            color=alpha_color,
            linestyle=ls,
            marker=mk,
            linewidth=1.5,
            markersize=4,
        )

        ax2.plot(
            split_x,
            d["theta_deg"],
            color=theta_color,
            linestyle=ls,
            marker=mk,
            linewidth=1.5,
            markersize=4,
        )

    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(season_order)
    ax1.set_xlabel("Season")
    ax1.set_ylabel(r"$\alpha$", color=alpha_color)
    ax2.set_ylabel(r"$\theta$ [$^\circ$]", color=theta_color)

    ax1.tick_params(axis="y", colors=alpha_color)
    ax2.tick_params(axis="y", colors=theta_color)

    # ax1.set_title(r"Seasonal $\alpha$ and $\theta$")
    ax1.grid(True, which="major", alpha=0.3)
    ax2.grid(False)

    parameter_handles = [
        Line2D([0], [0], color=alpha_color, lw=1.8, label=r"$\alpha$"),
        Line2D([0], [0], color=theta_color, lw=1.8, label=r"$\theta$"),
    ]

    split_handles = [
        Line2D(
            [0], [0],
            color="0.2",
            linestyle=style_map[split]["linestyle"],
            marker=style_map[split]["marker"],
            lw=1.5,
            markersize=4,
            label=split.capitalize(),
        )
        for split in splits if split in style_map
    ]

    # legend1 = ax1.legend(
    #     handles=parameter_handles,
    #     loc="upper left",
    #     title="Parameter",
    #     frameon=True,
    # )
    # ax1.add_artist(legend1)
    
    ax1.legend(
        handles=split_handles,
        handlelength=2.5,
        loc="center",
        bbox_to_anchor=(0.67, 0.85),
        title="Split",
        frameon=True,
    )

    ax1.set_ylim(0.01, 0.029)
    ax2.set_ylim(15, 55)


    sns.despine(ax=ax1, right=False)
    sns.despine(ax=ax2, left=False)

    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()

In [ ]:
plot_alpha_theta_by_season(df, height_ratio=0.5, save_path="alpha_theta_by_season.pdf")

In [ ]:
import numpy as np
import pandas as pd

def _scale_marker_sizes(values, smin=20, smax=120):
    values = pd.Series(values).astype(float)
    
    if values.nunique() <= 1:
        return pd.Series([0.5 * (smin + smax)] * len(values), index=values.index)
    
    vmin = values.min()
    vmax = values.max()
    
    scaled = smin + (values - vmin) * (smax - smin) / (vmax - vmin)
    return scaled

def plot_alpha_theta_by_season(
    df,
    splits=("train", "val", "test"),
    fontsize=9,
    height_ratio=0.62,
    scale_dots_by_samples=True,
    color_dots_by_samples=True,
    smin=20,
    smax=90,
    save_path=None,
):
    setup_pub_style(fontsize=fontsize)
    sns.set_style("whitegrid")

    palette = sns.color_palette("colorblind")
    alpha_color = palette[0]
    theta_color = palette[1]

    alpha_cmap = sns.light_palette(alpha_color, as_cmap=True)
    theta_cmap = sns.light_palette(theta_color, as_cmap=True)

    fig, ax1 = plt.subplots(figsize=fig_textwidth(height_ratio))
    ax2 = ax1.twinx()

    season_order = ["DJF", "MAM", "JJA", "SON"]

    style_map = {
        "train": {"linestyle": "-",  "marker": "o"},
        "val":   {"linestyle": "--", "marker": "s"},
        "test":  {"linestyle": ":",  "marker": "^"},
    }

    # First pass: compute fractions and collect all plotted fractions
    split_data = {}
    all_fractions = []

    for split in splits:
        d = df[(df["split"] == split) & (df["group_type"] == "season")].copy()
        if d.empty:
            continue

        d["season"] = pd.Categorical(d["season"], categories=season_order, ordered=True)
        d = d.sort_values("season").reset_index(drop=True)
        d = d.dropna(subset=["season"])

        if d.empty:
            continue
        if "n_samples_used" not in d.columns:
            raise KeyError("Dataframe must contain 'n_samples_used'.")

        split_total = d["n_samples_used"].sum()
        if split_total <= 0:
            continue

        d["sample_fraction"] = d["n_samples_used"] / split_total
        split_data[split] = d
        all_fractions.extend(d["sample_fraction"].tolist())

    if not split_data:
        raise ValueError("No seasonal data found for the requested splits.")

    # Use only plotted fractions for normalization
    vmin = min(all_fractions)
    vmax = max(all_fractions)

    # avoid degenerate normalization if all fractions are identical
    if vmax == vmin:
        vmax = vmin + 1e-12

    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    for split, d in split_data.items():
        x = [season_order.index(s) for s in d["season"].astype(str)]
        ls = style_map.get(split, {"linestyle": "-", "marker": "o"})["linestyle"]
        mk = style_map.get(split, {"linestyle": "-", "marker": "o"})["marker"]

        if scale_dots_by_samples:
            sizes = _scale_marker_sizes(d["n_samples_used"], smin=smin, smax=smax)
        else:
            sizes = pd.Series([35] * len(d), index=d.index)

        ax1.plot(
            x, d["alpha"],
            color=alpha_color,
            linestyle=ls,
            linewidth=1.5,
        )
        ax2.plot(
            x, d["theta_deg"],
            color=theta_color,
            linestyle=ls,
            linewidth=1.5,
        )

        if color_dots_by_samples:
            ax1.scatter(
                x, d["alpha"],
                s=sizes,
                c=d["sample_fraction"],
                cmap=alpha_cmap,
                norm=norm,
                marker=mk,
                edgecolors=alpha_color,
                linewidths=0.9,
                zorder=3,
            )
            ax2.scatter(
                x, d["theta_deg"],
                s=sizes,
                c=d["sample_fraction"],
                cmap=theta_cmap,
                norm=norm,
                marker=mk,
                edgecolors=theta_color,
                linewidths=0.9,
                zorder=3,
            )
        else:
            ax1.scatter(
                x, d["alpha"],
                s=sizes,
                color=alpha_color,
                marker=mk,
                edgecolors=alpha_color,
                linewidths=0.9,
                zorder=3,
            )
            ax2.scatter(
                x, d["theta_deg"],
                s=sizes,
                color=theta_color,
                marker=mk,
                edgecolors=theta_color,
                linewidths=0.9,
                zorder=3,
            )

    ax1.set_xticks(range(len(season_order)))
    ax1.set_xticklabels(season_order)
    ax1.set_xlabel("Season")
    ax1.set_ylabel(r"$\alpha$", color=alpha_color)
    ax2.set_ylabel(r"$\theta$ [$^\circ$]", color=theta_color)

    ax1.tick_params(axis="y", colors=alpha_color)
    ax2.tick_params(axis="y", colors=theta_color)

    ax1.grid(True, which="major", alpha=0.3)
    ax2.grid(False)

    split_handles = [
        Line2D(
            [0], [0],
            color="0.2",
            linestyle=style_map[split]["linestyle"],
            marker=style_map[split]["marker"],
            lw=1.5,
            markersize=4,
            label=split.capitalize(),
        )
        for split in splits if split in style_map and split in split_data
    ]

    ax1.legend(
        handles=split_handles,
        handlelength=2.5,
        loc="center",
        bbox_to_anchor=(0.67, 0.85),
        title="Split",
        frameon=True,
    )

    # gray_cmap = mpl.cm.Greys
    # if color_dots_by_samples:
    #     sm_gray = mpl.cm.ScalarMappable(norm=norm, cmap=gray_cmap)
    #     sm_gray.set_array([])

    #     cbar = fig.colorbar(
    #         sm_gray,
    #         ax=[ax1, ax2],
    #         orientation="horizontal",
    #         location="top",
    #         pad=0.02,
    #         fraction=0.08,
    #         aspect=35,
    #     )
    #     cbar.set_label("Fraction of split samples")

    ax1.set_ylim(0.01, 0.0295)
    ax2.set_ylim(15, 55)

    sns.despine(ax=ax1, right=False)
    sns.despine(ax=ax2, left=False)

    fig.tight_layout()#(rect=[0, 0, 1, 0.85])

    if save_path is not None:
        fig.savefig(save_path, bbox_inches="tight")
    plt.show()

In [ ]:
plot_alpha_theta_by_season(df, height_ratio=0.45, save_path="alpha_theta_by_season_size_hue.pdf")
